---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

---

## Group G35 — Pistar

- Ariel Cornelius Sitorus (13524085)
- Rava Khoman Tuah Saragih (13524107)
- Natanael I Manurung (13524021)
- Mahatma Brahmana (13524015)
- Raynard Fausta (13524052)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

Proyek ini menggunakan data booking hotel untuk memprediksi pembatalan.

## Overview

A canceled booking leaves an empty room that could have been sold to another guest. Early prediction of cancellations helps an accommodation property plan overbooking, send reminders, and set suitable cancellation policies.

This competition provides booking records from an accommodation property. Each record describes the guests, the length of stay, the meal package, the room category, the booking channel, the price, and the booking history of the guest.

The goal is a classification model, built from scratch, that predicts whether a booking is canceled (is_canceled = 1) or not canceled (is_canceled = 0).

## Aim

Tujuan proyek ini adalah memprediksi `is_canceled` untuk setiap booking. Nilai `1` berarti batal dan `0` berarti tidak batal. Kami membuat lima model klasifikasi from scratch dan membandingkannya dengan model scikit-learn.

## Metric

memakai Macro F1, yaitu rata-rata skor F1 untuk is_canceled. 

## Dataset

`train.csv` berisi 27.373 booking dengan label, sedangkan `test.csv` berisi 6.844 booking yang perlu diprediksi. Fiturnya mencakup informasi tamu, lama menginap, tipe kamar, paket makan, kanal booking, harga dalam rupiah, dan riwayat booking. Beberapa nilai fitur kosong. `sample_submission.csv` menunjukkan format `id,is_canceled`.

## Approach

Kami memakai pembagian train dan validation yang sama untuk K-Nearest Neighbors, C4.5, Logistic Regression, Support Vector Machine, dan Naive Bayes, serta pembandingnya dari scikit-learn. Preprocessing di-fit pada bagian train, lalu diterapkan ke validation. Model from scratch yang dipilih kemudian di-fit ulang pada seluruh data train yang sudah dibersihkan untuk menghasilkan satu prediksi per booking di test.

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

Instal dependensi dari `requirements.txt`, lalu simpan ketiga file CSV kompetisi di folder `data/`.

In [ ]:
# Jalankan dari folder utama proyek: python -m pip install -r requirements.txt

## Import Libraries

Impor library yang diperlukan untuk menyiapkan notebook dan membaca data.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)
from sklearn.utils.class_weight import compute_sample_weight

## Settings

Cari folder utama proyek supaya data bisa dibaca saat notebook dijalankan dari folder utama atau dari folder `notebooks/`.

In [ ]:
current_dir = Path.cwd().resolve()
for directory in (current_dir, *current_dir.parents):
    if (directory / 'requirements.txt').is_file():
        project_root = directory
        break
else:
    raise FileNotFoundError('Jalankan notebook dari dalam folder proyek.')


class Settings:
    SEED = 42
    DATA_DIR = project_root / 'data'
    OUTPUT_DIR = project_root / 'output'
    ID_COLUMN = 'id'

Settings.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

Tetapkan seed Python dan NumPy agar operasi acak dapat diulang.

In [ ]:
random.seed(Settings.SEED)
np.random.seed(Settings.SEED)

## Load Dataset

Baca ketiga file CSV kompetisi. Kolom `id` dibaca sebagai teks karena beberapa ID menyerupai angka.

In [ ]:
train = pd.read_csv(Settings.DATA_DIR / 'train.csv', dtype={Settings.ID_COLUMN: str})
test = pd.read_csv(Settings.DATA_DIR / 'test.csv', dtype={Settings.ID_COLUMN: str})
sample_submission = pd.read_csv(
    Settings.DATA_DIR / 'sample_submission.csv', dtype={Settings.ID_COLUMN: str}
)

pd.Series({'train': len(train), 'test': len(test), 'sample_submission': len(sample_submission)})

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
# Write your code here

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
# Write your code here

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
# Write your code here

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
TARGET = 'is_canceled'

train_set, val_set = train_test_split(
    train, test_size=0.2, stratify=train[TARGET], random_state=Settings.SEED
)
train_set = train_set.reset_index(drop=True)
val_set = val_set.reset_index(drop=True)
print(len(train_set), len(val_set))

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
# id dan target bukan fitur.
num_cols = [
    'num_adults', 'num_children', 'weekend_nights', 'weekday_nights',
    'needs_parking', 'days_before_arrival', 'arrival_year', 'arrival_month',
    'arrival_day', 'is_returning_guest', 'prior_cancellations',
    'prior_completed_bookings', 'nightly_rate_idr', 'special_requests',
]
cat_cols = ['meal_package', 'room_category', 'booking_channel']
feature_cols = num_cols + cat_cols

train_set[feature_cols].dtypes.value_counts()

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
# Nilai kosong per kolom di train. Pengisiannya (median) ada di pipeline, dihitung dari train saja.
missing = train_set[feature_cols].isna().sum()
missing[missing > 0]

### Duplicates

This part covers repeated records.

In [ ]:
# Duplikat dihitung tanpa kolom id dan hanya dibuang dari train.
dup_cols = feature_cols + [TARGET]
print('duplikat:', train_set.duplicated(subset=dup_cols).sum())
train_set = train_set.drop_duplicates(subset=dup_cols).reset_index(drop=True)
print('baris train:', len(train_set))

### Outliers

This part covers values far from the rest of the data.

In [ ]:
# Outlier tidak dibuang karena itu booking yang nyata. Kolom yang miring diredam log1p di pipeline.
skewed_cols = [
    'days_before_arrival', 'nightly_rate_idr',
    'prior_cancellations', 'prior_completed_bookings',
]
q1 = train_set[skewed_cols].quantile(0.25)
q3 = train_set[skewed_cols].quantile(0.75)
iqr = q3 - q1
outside = (train_set[skewed_cols] < q1 - 1.5 * iqr) | (train_set[skewed_cols] > q3 + 1.5 * iqr)
outside.sum()

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
# Nilai yang mustahil diganti NaN, nanti diisi imputer.
nonnegative = [
    'num_adults', 'num_children', 'weekend_nights', 'weekday_nights',
    'days_before_arrival', 'prior_cancellations', 'prior_completed_bookings',
    'nightly_rate_idr', 'special_requests',
]
binary_cols = ['needs_parking', 'is_returning_guest']
date_cols = ['arrival_year', 'arrival_month', 'arrival_day']


def fix_invalid(df):
    df = df.copy()
    for col in cat_cols:
        df[col] = df[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    for col in nonnegative:
        df.loc[df[col] < 0, col] = np.nan
    for col in binary_cols:
        df.loc[df[col].notna() & ~df[col].isin([0, 1]), col] = np.nan
    dates = pd.to_datetime(
        {'year': df['arrival_year'], 'month': df['arrival_month'], 'day': df['arrival_day']},
        errors='coerce',
    )
    df.loc[dates.isna(), date_cols] = np.nan
    return df


before = train_set[feature_cols].isna().sum().sum()
train_set = fix_invalid(train_set)
val_set = fix_invalid(val_set)
print('sel yang jadi NaN di train:', train_set[feature_cols].isna().sum().sum() - before)

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

This part covers new attributes derived from existing ones.

In [ ]:
# Tiga fitur turunan, tidak memakai target.
def add_features(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    return df


train_set = add_features(train_set)
val_set = add_features(val_set)
engineered = ['total_nights', 'total_guests', 'is_free']
num_all = num_cols + engineered

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
# Kolom miring di-log dulu, lalu semua kolom angka distandardisasi (rata-rata dan std dari train).
def log_clip(x):
    return np.log1p(np.clip(x, 0, None))


plain_cols = [col for col in num_all if col not in skewed_cols]

skewed_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(log_clip)),
    ('scale', StandardScaler()),
])
plain_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
# matrix (KNN, LogReg, SVM): one-hot, kategori kosong jadi kolomnya sendiri,
# angka kosong diberi kolom penanda. tree (DT, NB): kategori jadi kode angka, kosong tetap NaN.
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
flags = MissingIndicator(features='missing-only', error_on_new=False)

matrix_columns = ColumnTransformer([
    ('skewed', skewed_pipe, skewed_cols),
    ('plain', plain_pipe, plain_cols),
    ('flags', flags, num_all),
    ('cat', onehot, cat_cols),
])
tree_columns = ColumnTransformer([
    ('num', 'passthrough', num_all),
    ('cat', ordinal, cat_cols),
])

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
# Cek apakah dimensi perlu dikurangi (hanya memakai train).
probe = matrix_columns.fit_transform(train_set)
y_probe = train_set[TARGET].to_numpy(dtype=int)

pca = PCA().fit(probe)
needed = np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.95) + 1
print(f'{probe.shape[1]} fitur, {needed} komponen PCA untuk 95% varians')

mi = mutual_info_classif(probe, y_probe, random_state=Settings.SEED)
print('fitur dengan MI di bawah 0.001:', int((mi < 0.001).sum()))

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
print(train_set[TARGET].value_counts(normalize=True).round(3))

# Ketidakseimbangan ditangani lewat bobot sampel, bukan dengan membuang baris.
sample_weight = compute_sample_weight('balanced', train_set[TARGET])

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
y_train = train_set[TARGET].to_numpy(dtype=int)
y_val = val_set[TARGET].to_numpy(dtype=int)
print(y_train.shape, y_val.shape, sample_weight.shape)

---

# Preparation Pipeline <a name="6"></a>

---

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
# Cleaning dan fitur turunan ikut di pipeline, jadi data mentah (misalnya test) lewat langkah yang sama.
def clean_features(df):
    return add_features(fix_invalid(df))


prep_matrix = Pipeline([('clean', FunctionTransformer(clean_features)), ('columns', matrix_columns)])
prep_tree = Pipeline([('clean', FunctionTransformer(clean_features)), ('columns', tree_columns)])

X_train_matrix = prep_matrix.fit_transform(train_set)
X_val_matrix = prep_matrix.transform(val_set)
X_train_tree = prep_tree.fit_transform(train_set)
X_val_tree = prep_tree.transform(val_set)
tree_cat_idx = list(range(len(num_all), len(num_all) + len(cat_cols)))

print('matrix:', X_train_matrix.shape, '| tree:', X_train_tree.shape)

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

This part covers the nearest neighbor classifier and its validation.

In [ ]:
# Write your code here

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
# Write your code here

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# Write your code here

## Support Vector Machine

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
# Write your code here

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
# Write your code here

## Model Evaluation

This part compares the validation results of all models.

In [ ]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
# Write your code here

---

# Saving and Loading Models <a name="9"></a>

---

This section stores the trained models and restores them for prediction.

In [ ]:
# Write your code here

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
# Write your code here

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.